# Previsão de Preços no E-commerce Brasileiro com Machine Learning: Base Olist

Machine Learning para Predição de Preços no E-commerce

Neste projeto, vou desenvolver um modelo de Machine Learning para estimar o preço de produtos em um marketplace. O projeto envolve aprendizado supervisionado para um problema de regressão, no qual a variável-alvo será o preço do produto.

Utilizarei a base pública da Olist, um marketplace brasileiro, contendo informações sobre produtos, vendedores, pedidos e outras características que podem ajudar a explicar as diferenças de preços.

Vou utilizar o seguinte pipeline:

1. Definição do problema de negócio.
2. Limpeza e pré-processamento dos dados.
3. Entendimento do problema e definição da estratégia de validação.
4. Divisão dos dados em conjuntos de treino e teste.
5. Exploração dos dados (EDA).
6. Treinamento dos modelos e ajuste de hiperparâmetros
7. Teste e avaliação final do modelo.
8. Interpretação dos resultados e geração de insights de negócio.
9. Deploy do modelo.

Neste notebook, realizaremos as etapas de 1 a 3 do projeto, com foco na definição do problema e na limpeza dos dados direcionada aos objetivos dessa análise.

## 1. Problema de negócio

Estimar o preço de produtos em um marketplace.

A precificação refere-se ao processo de definir quanto cobrar por um produto ou serviço. É uma das decisões mais importantes para um negócio, pois influencia diretamente a demanda, a receita e a margem de lucro.

A definição adequada de preços é importante para um marketplace por diversos motivos:

1. Competitividade: preços muito acima do mercado podem afastar clientes, enquanto preços competitivos podem aumentar a atratividade dos produtos.
2. Rentabilidade: preços muito baixos podem reduzir as margens de lucro e deixar receita potencial na mesa.
3. Tomada de decisão: uma estimativa de preço baseada em dados pode auxiliar vendedores e empresas na definição de preços, reduzindo a dependência de decisões puramente manuais.
4. Escalabilidade: em um marketplace com milhares de produtos e diferentes categorias, definir preços individualmente pode ser trabalhoso. Modelos preditivos podem ajudar a automatizar parte desse processo.
5. Compreensão dos fatores de preço: a análise dos dados permite identificar quais características dos produtos estão mais associadas ao seu valor, como categoria, dimensões e outras características disponíveis na base.
6. Apoio à estratégia de pricing: entender os padrões de preços pode servir como base para estratégias mais avançadas de precificação e otimização.

Ao utilizar modelos preditivos, é possível estimar o preço de um produto a partir de suas características e obter insights sobre os principais fatores associados à precificação. Dessa forma, o objetivo deste projeto é desenvolver um modelo capaz de fornecer uma estimativa de preço e, ao mesmo tempo, contribuir para uma melhor compreensão da dinâmica de preços no marketplace.

### importando as bibliotecas

In [0]:
%pip install seaborn
import pandas as pd
import numpy as np
import seaborn as sns
import zipfile
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import unicodedata

## 2. Limpeza e pré-processamento dos dados

A base de dados foi coletada do Kagle :  https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [0]:
itens = spark.table("olist.silver.order_items")
orders = spark.table("olist.silver.orders")
produtos = spark.table("olist.silver.products")
vendedores = spark.table("olist.silver.sellers")

# juntando as bases
order_data = (
    itens
    .join(orders, "order_id", "left")
    .join(produtos, "product_id", "left")
    .join(vendedores, "seller_id", "left")
)
print(itens.count(), order_data.count())   # devem ser iguais

# remove itens sem categoria de produto
nulos = order_data.filter(order_data.product_category_name.isNull()).count()
print(nulos, nulos / order_data.count())
order_data = order_data.dropna(subset=["product_category_name"])
print(order_data.count(), len(order_data.columns))

Foram identificadas 1.603 observações sem categoria, correspondendo a aproximadamente 1,42% da base. Optei por removê-las, pois a categoria é uma variável importante para a previsão de preços e a perda de dados é pequena. Além disso, como todo produto deveria possuir uma categoria, esses valores provavelmente representam falhas ou inconsistências no registro dos dados. Para esse projeto, assumirei que todos os futuros produtos terão Categoria

### retirando colunas não necessárias

In [0]:
order_data.columns

In [0]:
cols_remover = [
    # identificadores: não carregam informação de preço e, com alta
    # cardinalidade, favorecem overfitting
    'order_id', 'customer_id','order_item_id',

    # pós-venda: não existiriam no momento de definir o preço
    'order_status',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',

    # logística do pedido: dependem do pedido e do frete, não do produto
    'shipping_limit_date',
    'order_estimated_delivery_date',
]
order_data = order_data.drop(*cols_remover)
order_data = order_data.toPandas()
order_data.shape   

## 3. Entendimento do problema e definição da estratégia de validação

Uma questão importante é avaliar se seller_id deve ser considerada uma feature. Alguns vendedores podem utilizar estratégias como clubes de desconto, cupons, promoções, entre outras, que podem influenciar os preços praticados.

Além disso, é importante verificar o comportamento do product_id, pois os mesmos produtos podem ter sido vendidos repetidamente ao longo do período. A seguir, irei analisar essas características para avaliar se devem ser utilizadas no modelo.

In [0]:
colunas = ['seller_id', 'product_id']

for coluna in colunas:
    print("\n" + "=" * 50)
    print(f"ANÁLISE: {coluna.upper()}")
    print("=" * 50)

    n_unicos = order_data[coluna].nunique()
    vc = order_data[coluna].value_counts()

    print(f"Existem {n_unicos} {coluna} únicos")
    print(f"{(vc < 5).sum()} com menos de 5 ocorrências")
    
    print("\n4 maiores:")
    print(vc.head(4))
    
    print("\nEstatísticas:")
    print(vc.describe())

Optei, inicialmente, por manter seller_id e product_id como possíveis features, pois ambos possuem informações que podem contribuir para a previsão dos preços. No entanto, as duas variáveis apresentam alta cardinalidade e muitos valores com poucas ocorrências, exigindo um tratamento adequado para reduzir o risco de overfitting.

Dessa forma, serão avaliadas técnicas de codificação apropriadas para essas variáveis. Também é viável desenvolver modelos sem elas, permitindo verificar posteriormente o quanto realmente contribuem para a previsão.

In [0]:
order_data.info()

In [0]:
order_data[order_data['product_weight_g'].isna()]

In [0]:
#irei dropar essa unica linha sem informaçoes do peso e das medidas do produto
order_data = order_data.dropna(subset=['product_weight_g'])

In [0]:
# Convertendo de volta para o formato Delta
spark.createDataFrame(order_data).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.gold.projeto_pricing")